# QTradeSystematic - ML-Factor End-to-End Flow
End-to-end: data load -> feature engineering -> XGB model -> backtest -> CSV + pyfolio tearsheet export.


In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from datetime import date, datetime
from decimal import Decimal
from functools import partial
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import yaml

from qts.config.builder import Config
from qts.core.registry import Registry
from qts.orchestration.runtime import build_data_manager
from qts.research.backtest.base import BacktestConfig
from qts.research.backtest.engines.vectorbtpro_engine import VectorBTProEngine
from qts.research.backtest.pyfolio_adapter import (
    positions_frame,
    returns_series,
    transactions_frame,
)
from qts.research.backtest.tearsheet import save_tearsheet
from qts.research.features.pipeline import FeaturePipeline
from qts.utils.export import export_portfolio_snapshots, export_trade_log
from qts.utils.paths import backtest_exports_dir, tearsheet_dir

print("Imports OK")


## Step 1 - Load YAML config


In [ ]:
CONFIG_PATH = Path("../configs/strategies/ml_factor/vn100_ml_example.yaml")

resolved = Config.build(str(CONFIG_PATH))
config: BacktestConfig = resolved.raw

symbols = [
    *config.universe.stock,
    *config.universe.vn_stock,
    *config.universe.vn_warrant,
    *config.universe.vn_futures,
    *config.universe.crypto,
    *config.universe.crypto_futures,
]

print(f"Workflow       : {config.workflow}")
print(f"Universe       : {symbols}")
print(f"Date range     : {config.start_date} -> {config.end_date}")
print(f"Test start     : {config.test_start_date}")
print(f"Initial capital: {config.initial_capital:,}")
print(f"Engine         : {config.backtest_engine}")
print(f"Benchmark      : {config.benchmark}")


## Step 2 - Fetch OHLCV from DuckDB


In [ ]:
data_manager = build_data_manager(resolved)

raw: pl.DataFrame = await data_manager.get_ohlcv(
    symbols=symbols,
    start=config.start_date,
    end=config.end_date,
)

print(f"Rows: {len(raw):,}  |  Symbols: {raw['symbol'].n_unique()}  |  Columns: {raw.columns}")
raw.head(5)


## Step 3 - Feature Engineering


In [ ]:
feature_pipeline: FeaturePipeline = resolved.feature_pipeline
featured: pl.DataFrame = feature_pipeline.fit_transform(raw)

print(f"Featured shape : {featured.shape}")
print(f"Feature columns: {[col for col in featured.columns if col not in raw.columns]}")
featured.head(3)


## Step 4 - Feature exploration


In [ ]:
predictor_cols = config.strategy.params["predictor_cols"]
target_col = config.strategy.params["target_col"]

print(f"Predictors : {predictor_cols}")
print(f"Target     : {target_col}")

null_rates = {
    col: featured[col].null_count() / len(featured)
    for col in predictor_cols + [target_col]
}
print("\nNull rates:")
for col, rate in null_rates.items():
    print(f"  {col:40s}  {rate:.1%}")

featured.select(predictor_cols).describe()


In [ ]:
import numpy as np

corr = featured.select(predictor_cols).to_pandas().corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(predictor_cols)))
ax.set_yticks(range(len(predictor_cols)))
ax.set_xticklabels(predictor_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(predictor_cols, fontsize=9)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()


## Step 5 - Build XGB model + MLFactorStrategy


In [ ]:
from qts.research.strategies.ml_factor.model import MLFactorStrategy
from qts.research.strategies.ml_factor.models import XGBClassifierModel

model_params = config.strategy.params.get("model", {}).get("params", {})
model = XGBClassifierModel(task="classification", **model_params)

portfolio_cfg = config.portfolio_construction
portfolio_func = None
if portfolio_cfg is not None:
    portfolio_func = partial(
        Registry.get_portfolio_constructor(portfolio_cfg.name),
        **portfolio_cfg.params,
    )

strategy = MLFactorStrategy(
    predictor_cols=predictor_cols,
    target_col=target_col,
    model=model,
    portfolio_func=portfolio_func,
    task=config.strategy.params.get("task", "classification"),
    rebalance_period=config.strategy.params.get("rebalance_period", 21),
)
print(f"Strategy: {strategy}")


## Step 6 - Run backtest (VectorBTProEngine)


In [ ]:
engine = VectorBTProEngine()

result = engine.run(
    strategy=strategy,
    data=featured,
    config=config,
    pipeline=feature_pipeline,
    ohlcv=raw,
)

print("\n=== In-sample metrics ===")
for key, value in result.metrics.items():
    print(f"  {key:20s}: {value:.4f}")


## Step 7 - In-sample vs out-of-sample split check


In [ ]:
oos_start = pd.Timestamp(config.test_start_date).tz_localize("UTC")
rets_series = returns_series(result)

is_rets = rets_series[rets_series.index < oos_start]
oos_rets = rets_series[rets_series.index >= oos_start]

def annualised_sharpe(series: pd.Series) -> float:
    if series.empty or series.std() == 0:
        return 0.0
    return (series.mean() / series.std()) * (252 ** 0.5)

def period_label(series: pd.Series) -> str:
    if series.empty:
        return "empty"
    return f"{series.index[0].date()} -> {series.index[-1].date()}"

print(f"IS  period : {period_label(is_rets)}  |  Sharpe: {annualised_sharpe(is_rets):.3f}")
print(f"OOS period : {period_label(oos_rets)}  |  Sharpe: {annualised_sharpe(oos_rets):.3f}")

fig, ax = plt.subplots(figsize=(12, 4))
equity = result.equity_curve.to_pandas()
equity["date"] = pd.to_datetime(equity["date"])
ax.plot(
    equity["date"],
    equity["equity"] / equity["equity"].iloc[0],
    label="Strategy",
    linewidth=1.5,
)
if config.test_start_date:
    ax.axvline(
        pd.Timestamp(config.test_start_date),
        color="red",
        linestyle="--",
        alpha=0.7,
        label="OOS start",
    )
ax.set_title("Equity curve (normalised)")
ax.set_ylabel("Growth of $1")
ax.legend()
plt.tight_layout()
plt.show()


## Step 8 - Save CSV outputs


In [ ]:
run_id = f"{result.engine_name}_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}"
exports_dir = backtest_exports_dir()

tl_path = exports_dir / f"{run_id}_trade_log.csv"
snp_path = exports_dir / f"{run_id}_snapshots.csv"

export_trade_log(result, tl_path)
export_portfolio_snapshots(result, snp_path)

print(f"Trade log  -> {tl_path}  ({result.trade_log.shape[0]} rows)")
print(f"Snapshots  -> {snp_path}  ({result.portfolio_snapshots.shape[0]} rows)")
result.trade_log.head(5)


## Step 9 - Generate pyfolio tearsheet


In [ ]:
benchmark_rets = None
if config.benchmark:
    from qts.orchestration.flow import _fetch_benchmark_returns

    benchmark_rets = _fetch_benchmark_returns(
        config.benchmark,
        config.start_date,
        config.end_date,
        data_manager,
    )
    if benchmark_rets is not None:
        print(f"Benchmark loaded: {len(benchmark_rets)} days of {config.benchmark}")
    else:
        print(f"Benchmark {config.benchmark} not found in DB - running without")


In [ ]:
pdf_path = save_tearsheet(
    result=result,
    out_dir=tearsheet_dir(),
    run_id=run_id,
    benchmark_rets=benchmark_rets,
)

if pdf_path:
    print(f"Tearsheet PDF -> {pdf_path}  ({pdf_path.stat().st_size / 1024:.1f} KB)")
else:
    print("Tearsheet skipped (pyfolio not installed)")


In [ ]:
if pdf_path and pdf_path.exists():
    import shutil
    import subprocess

    if shutil.which("pdftoppm"):
        png_out = pdf_path.with_suffix("")
        subprocess.run(
            ["pdftoppm", "-r", "150", "-png", "-l", "1", str(pdf_path), str(png_out)],
            check=False,
        )
        candidates = list(pdf_path.parent.glob(f"{run_id}_tearsheet-*.png"))
        if candidates:
            from IPython.display import Image, display

            display(Image(str(candidates[0])))
        else:
            print(f"Open the PDF manually: {pdf_path}")
    else:
        print(f"Install poppler for inline display. PDF at: {pdf_path}")


## Summary

| Output | Path |
|--------|------|
| Trade log CSV | `~/.qts/exports/backtest/{run_id}_trade_log.csv` |
| Snapshots CSV | `~/.qts/exports/backtest/{run_id}_snapshots.csv` |
| Tearsheet PDF | `~/.qts/exports/backtest/tearsheets/{run_id}_tearsheet.pdf` |
